# Chapter 4 — Memory: Giving the DevOps Agent a Memory of Past Incidents

Companion notebook to the seven `skills/memory/` primitives. This is the
Chapter 4 *"Adding Memory to the DevOps Agent"* section made runnable.

**The running scenario.** A DevOps agent watches the checkout API on the
fictional AWS account `123456789012`. The checkout API keeps having 503 /
connection-pool incidents — the latest one is an outage at
`2026-03-15T08:00Z`. In Chapter 3 the agent had a *snapshot* graph of the
current infrastructure. That snapshot cannot answer the questions an incident
actually raises:

- *What config was running **at the moment of the outage**, not what it is now?*
- *What changed in the hours before the outage?*
- *Have we seen this failure shape before — is it a known pattern?*
- *When the agent says "the root cause is v3.5.0", how does it know that?*

Chapter 4's thesis: **memory is not a log, it is a graph that evolves over
time.** This notebook threads one incident dataset through all seven memory
primitives, in the order the chapter builds them:

| # | Skill | Chapter idea | What it buys the agent |
|---|-------|--------------|------------------------|
| 1 | `bi-temporal-edge` | Temporal Awareness / Ex. 4-2 | answer *"what was true at outage time"* |
| 2 | `graphiti-incremental-update` | Graphiti pattern / Ex. 4-8 | ingest new telemetry without re-processing the whole graph |
| 3 | `hierarchical-memory` | Letta/MemGPT / Ex. 4-6 | keep durable facts hot, evict short-lived noise (but keep it searchable) |
| 4 | `hindsight-epistemic-classifier` | Hindsight four networks | separate *observed* from *believed* — "how do you know that?" |
| 5 | `letta-failure-modes` | Letta Leaderboard eight failures | diagnose the memory architecture we built |
| 6 | `memory-consolidation` | Consolidation / Ex. 4-5 | turn three noisy incidents into one durable **Pattern** |
| 7 | `rrf-hybrid-retrieval` | RRF (Latimer et al. 2025) / Retrieval | merge four retrieval channels into one ranked context |

Every AWS call is mocked with `moto` against the fictional account — the
notebook runs top-to-bottom with **zero real credentials**. Each section
teaches the *why* first (a markdown cell), then exercises the skill, then
verifies with an assert. The final cell composes all seven into the temporal
root-cause query the memory can now answer.

## 0. Setup — a clean way to load each skill's `lib.py`

Every skill ships a `lib.py` (the importable library) and a `cli.py` (the
multi-harness command surface). Several skills each define a module named
`lib`, so a plain `import lib` would collide in `sys.modules`. We load each
skill's library under a unique name with `importlib`, which keeps the seven
primitives cleanly separated. We also demonstrate the `cli.py` surface via a
subprocess `--help` call (the CLI mandate: `--help` prints the skill's own
description).

In [ ]:
import os, sys, json, subprocess, importlib.util, tempfile
from pathlib import Path
from datetime import datetime, timedelta, timezone

# Locate the repo and the memory-skills directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SKILLS = PROJECT_ROOT / "skills" / "memory"
assert SKILLS.is_dir(), f"memory skills not found at {SKILLS}"


def load_skill_lib(skill_name):
    """Load skills/memory/<skill_name>/lib.py under a unique module name."""
    lib_path = SKILLS / skill_name / "lib.py"
    mod_name = "skilllib_" + skill_name.replace("-", "_")
    spec = importlib.util.spec_from_file_location(mod_name, lib_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    spec.loader.exec_module(mod)
    return mod


def run_cli(skill_name, *args):
    """Invoke a skill's cli.py as a subprocess (harness-portability path)."""
    cli = SKILLS / skill_name / "cli.py"
    proc = subprocess.run(
        [sys.executable, str(cli), *args],
        capture_output=True, text=True, cwd=str(SKILLS / skill_name),
    )
    return proc


# moto: mock AWS so the notebook runs with zero real credentials
os.environ["AWS_ACCESS_KEY_ID"] = "testing"
os.environ["AWS_SECRET_ACCESS_KEY"] = "testing"
os.environ["AWS_SECURITY_TOKEN"] = "testing"
os.environ["AWS_SESSION_TOKEN"] = "testing"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

import boto3
from moto import mock_aws

FICTIONAL_ACCOUNT = "123456789012"
OUTAGE_TS = datetime.fromisoformat("2026-03-15T08:00:00+00:00")

# CLI mandate check: cli.py --help exits 0 and prints the skill description
help_proc = run_cli("bi-temporal-edge", "--help")
assert help_proc.returncode == 0, help_proc.stderr
assert "bi-temporal" in help_proc.stdout.lower()
print(f"setup ok — fictional AWS account {FICTIONAL_ACCOUNT}, outage at {OUTAGE_TS.isoformat()}")
print("cli --help surface confirmed (first line):")
print("   " + help_proc.stdout.strip().splitlines()[0][:100])

## 1. Bi-temporal edges — *"what was true at the moment of the outage?"*

A memory that overwrites facts in place loses the historical trace that makes
incident reconstruction possible. By the time the post-mortem starts, the
config store says the checkout API runs `m5.xlarge` — but that migration
happened on 2026-03-10, *before* the outage, and the agent needs to know the
state **at outage time**, not now.

Chapter 4's Example 4-2 gives every relationship **two** independent time
dimensions:

- **Validity time** (`valid_from` / `valid_until`) — when the fact was true in
  the domain. `valid_until=None` means *currently valid*.
- **Ingestion time** (`ingested_at`) — when the *system learned* the fact. It
  can lag the validity by days (a staging change logged late). The lag is what
  debugs *"was the agent acting on stale data?"*

Once edges carry both, `was_valid_at(ts)` and `as_of(source, rel, ts)` turn
point-in-time reconstruction from forensic guesswork into a one-line query.

In [ ]:
bt = load_skill_lib("bi-temporal-edge")

# Build the checkout-api config timeline by hand, with historical dates.
# create_edge lets us backfill valid_from / ingested_at; we close each prior
# window with an explicit valid_until + invalidation_reason (the audit trail).
edges = []

def historical(rel, value, valid_from, ingested_at, link_type="entity"):
    return bt.create_edge(
        "service-checkout-api", "aws-ec2", rel, value=value,
        valid_from=datetime.fromisoformat(valid_from),
        ingested_at=datetime.fromisoformat(ingested_at),
        link_type=link_type,
        metadata={"aws_account": FICTIONAL_ACCOUNT, "region": "us-east-1"},
    )

# instance_type: t3.large (Feb 15) -> m5.xlarge (Mar 10, still current)
e_t3 = historical("instance_type", "t3.large", "2026-02-15T14:23:00+00:00", "2026-02-15T14:25:00+00:00")
e_m5 = historical("instance_type", "m5.xlarge", "2026-03-10T17:45:00+00:00", "2026-03-10T17:46:00+00:00")
e_t3.valid_until = e_m5.valid_from
e_t3.invalidation_reason = "next-gen migration to m5 family for steady-state throughput"
edges += [e_t3, e_m5]

# deployment_version: v3.4.1 (Mar 01) -> v3.5.0 (Mar 14 22:30 — the change
# that precedes the 08:00 outage). causal link_type so HINDSIGHT traversal
# weights it above weak-semantic neighbours.
e_v341 = historical("deployment_version", "v3.4.1", "2026-03-01T09:00:00+00:00", "2026-03-01T09:00:00+00:00", link_type="causal")
e_v350 = historical("deployment_version", "v3.5.0", "2026-03-14T22:30:00+00:00", "2026-03-14T22:31:00+00:00", link_type="causal")
e_v341.valid_until = e_v350.valid_from
e_v341.invalidation_reason = "superseded by v3.5.0 release"
edges += [e_v341, e_v350]

# A late-ingested fact: an EBS volume resize that was VALID on Jan 18 but only
# LOGGED on Mar 15 — a 56-day ingestion lag the agent must surface.
e_ebs = historical("ebs_volume_gb", "500", "2026-01-18T00:00:00+00:00", "2026-03-15T09:10:00+00:00")
edges.append(e_ebs)

# --- Point-in-time queries at the outage timestamp -------------------------
inst_at_outage = bt.as_of("service-checkout-api", "instance_type", OUTAGE_TS, edges)
deploy_at_outage = bt.as_of("service-checkout-api", "deployment_version", OUTAGE_TS, edges)
print(f"At the outage ({OUTAGE_TS.isoformat()}):")
print(f"   instance_type      = {inst_at_outage.value}")
print(f"   deployment_version = {deploy_at_outage.value}")

# What changed in the 24h window before the outage?
window_start = OUTAGE_TS - timedelta(hours=24)
recent_changes = [
    e for e in bt.history("service-checkout-api", edges)
    if window_start <= e.valid_from < OUTAGE_TS
]
print(f"\nChanges in the 24h before the outage:")
for e in recent_changes:
    print(f"   [{e.valid_from.isoformat()}] {e.relationship} -> {e.value}  ({e.link_type})")

# Ingestion lag: flag facts the agent learned about late
print(f"\nIngestion lag on the EBS fact: {e_ebs.ingestion_lag()}  "
      f"(valid {e_ebs.valid_from.date()}, learned {e_ebs.ingested_at.date()})")

# --- Verification gate -----------------------------------------------------
assert inst_at_outage.value == "m5.xlarge", "instance at outage should be m5.xlarge (migrated Mar 10)"
assert deploy_at_outage.value == "v3.5.0", "deploy at outage should be v3.5.0"
# Before the migration the instance was t3.large:
before = datetime.fromisoformat("2026-03-01T00:00:00+00:00")
assert bt.as_of("service-checkout-api", "instance_type", before, edges).value == "t3.large"
# The only change in the 24h window is the v3.5.0 deploy:
assert any(e.value == "v3.5.0" for e in recent_changes) and len(recent_changes) == 1
assert e_ebs.ingestion_lag() > timedelta(days=55), "EBS fact should show a ~56-day ingestion lag"
print("\n[PASS] bi-temporal edges answer point-in-time, change-window, and ingestion-lag queries.")

### 1b. Ground the memory in a **mocked** CloudWatch incident timeline

The bi-temporal edges above are what the agent *remembers*. Where do the raw
signals come from? From AWS, at incident time. Here we mock CloudWatch Logs
(the account is fictional, `moto` supplies real boto3 shapes) to seed the
08:00 latency spike, and mock EC2 `describe-instances` to confirm that the
*current* instance type the cloud reports matches what our bi-temporal
`as_of(now)` says. Same boto3 code you would run in production — no
credentials, no charges.

In [ ]:
import time as _time

@mock_aws
def seed_and_verify_incident():
    # 1. CloudWatch Logs: seed the checkout-api log stream with the 08:00 spike
    logs = boto3.client("logs")
    log_group = "/aws/ecs/checkout-api"
    logs.create_log_group(logGroupName=log_group)
    logs.create_log_stream(logGroupName=log_group, logStreamName="task-outage-0315")
    base = int(OUTAGE_TS.timestamp() * 1000)
    events = [
        {"timestamp": base - 600_000, "message": "INFO checkout duration_ms=41 status=200"},
        {"timestamp": base - 120_000, "message": "WARN checkout duration_ms=2810 status=200 db_pool_wait_ms=2700"},
        {"timestamp": base,           "message": "ERROR checkout duration_ms=5031 status=504 db_pool_exhausted=true"},
        {"timestamp": base + 60_000,  "message": "ERROR checkout duration_ms=4998 status=504 db_pool_exhausted=true"},
    ]
    logs.put_log_events(logGroupName=log_group, logStreamName="task-outage-0315", logEvents=events)
    q = logs.start_query(
        logGroupName=log_group,
        startTime=int(OUTAGE_TS.timestamp()) - 3600,
        endTime=int(OUTAGE_TS.timestamp()) + 3600,
        queryString="fields @timestamp,@message | filter status=504 | sort @timestamp desc",
    )

    # 2. EC2: the cloud's CURRENT instance type for checkout-api
    ec2 = boto3.client("ec2", region_name="us-east-1")
    images = ec2.describe_images()["Images"]
    ami = images[0]["ImageId"] if images else "ami-12345678"
    run = ec2.run_instances(
        ImageId=ami, MinCount=1, MaxCount=1, InstanceType="m5.xlarge",
        TagSpecifications=[{"ResourceType": "instance",
                            "Tags": [{"Key": "Name", "Value": "service-checkout-api"},
                                     {"Key": "AwsAccount", "Value": FICTIONAL_ACCOUNT}]}],
    )
    iid = run["Instances"][0]["InstanceId"]
    desc = ec2.describe_instances(InstanceIds=[iid])
    cloud_type = desc["Reservations"][0]["Instances"][0]["InstanceType"]
    return {"query_id": q["queryId"], "events": len(events), "cloud_instance_type": cloud_type}

incident = seed_and_verify_incident()
now = datetime.now(timezone.utc)
remembered_now = bt.as_of("service-checkout-api", "instance_type", now, edges).value
print(f"CloudWatch Logs seeded {incident['events']} events, queryId={incident['query_id'][:8]}...")
print(f"EC2 describe-instances (mocked): current instance_type = {incident['cloud_instance_type']}")
print(f"bi-temporal as_of(now):          remembered instance_type = {remembered_now}")

assert isinstance(incident["query_id"], str) and incident["query_id"]
assert incident["cloud_instance_type"] == remembered_now, \
    "the cloud's current instance type must match what the memory remembers as currently-valid"
print("\n[PASS] the mocked CloudWatch/EC2 signals agree with the bi-temporal memory's current state.")

## 2. Graphiti incremental update — ingest telemetry **without reprocessing the whole graph**

New operational events never stop arriving. Zep's Graphiti pattern (Example
4-8) keeps write latency predictable by doing everything **incrementally**:
extract entities from *only the new episode*, resolve them against existing
nodes (so `Sarah` mentioned twice becomes one node, not two), and touch only
the impacted neighbourhood. The rest of the graph is untouched.

The load-bearing property is **locality**: the number of nodes an episode
touches must be independent of how large the graph already is. We prove that
below by feeding the *same* episode to a small graph and a large one and
checking the touch-count is identical.

In [ ]:
gi = load_skill_lib("graphiti-incremental-update")

g = gi.Graph()
incident_stream = [
    "person:Sarah(s.chen) opened incident:INC-1042 affecting service:checkout-api in region:us-east-1",
    "deployment:v3.5.0 of service:checkout-api went live at 22:30 UTC",
    "deployment:v3.5.0 introduced 504 timeouts on service:payments calls",
    "person:Marcus joined incident:INC-1042 as secondary on-call",
    "person:Sarah triggered rollback to deployment:v3.4.1 of service:checkout-api",
    "person:Sarah(s.chen) closed incident:INC-1042 after rollback succeeded",
]
print("Streaming incident episodes (touched nodes + % of graph changed per episode):")
for i, ep in enumerate(incident_stream, 1):
    before = g.snapshot()
    log = gi.add_episode(ep, g, episode_id=f"ep-{i:03d}")
    after = g.snapshot()
    loc = gi.verify_locality(before, after)
    print(f"   ep{i:02d}: touched={log['touched_nodes']:2d}  changed={loc['percent_changed']:5.1f}%  "
          f"methods={log['resolution_method_counts']}")
print(f"\nFinal graph: {len(g.nodes)} nodes, {len(g.edges)} edges")

# Entity resolution dedup: 'Sarah' appears in 3 episodes but is ONE node
# (g.nodes is keyed by UUID; inspect the Node objects' names)
sarah_nodes = [n.name for n in g.nodes.values() if "sarah" in n.name.lower()]
print(f"'Sarah' node(s) after 3 mentions: {sarah_nodes}  (deduplicated to one)")

# Locality invariant: the same episode touches the same node count regardless
# of graph size.
g_small = gi.Graph()
for k in range(5):
    g_small.add_node(f"svc-{k}", "service")
g_big = gi.Graph()
for k in range(500):
    g_big.add_node(f"svc-{k}", "service")
touched_small = gi.add_episode("person:Eve opened incident:INC-9", g_small)["touched_nodes"]
touched_big = gi.add_episode("person:Eve opened incident:INC-9", g_big)["touched_nodes"]
print(f"\nSame episode: touched {touched_small} in a 5-node graph, {touched_big} in a 500-node graph")

# --- Verification gate -----------------------------------------------------
assert len(sarah_nodes) == 1, "entity resolution should dedup Sarah to a single node"
assert touched_small == touched_big, "locality violated: touch-count must be independent of graph size"
# The last Sarah mention canonical-resolves (she already exists in the graph):
last = gi.add_episode("person:Sarah reviewed incident:INC-1042 postmortem", g)
assert last["resolution_method_counts"].get("canonical", 0) >= 1, "re-mentioned Sarah should canonical-resolve"
print("\n[PASS] incremental updates are local and de-duplicate entities across episodes.")

## 3. Hierarchical memory — keep durable facts hot, evict the noise (but keep it searchable)

Chapter 4 maps agent memory onto the CPU memory hierarchy (Example 4-6):

- **core** — the small, fast working set that feeds the LLM's context window
  (the scarcest resource). `core_limit` is a *forcing function*, not a tuning
  knob: once full, something must be evicted.
- **recall** — raw interaction history (*"what did we talk about yesterday?"*).
- **archival** — effectively unlimited, slower, still searchable.

The key distinction is **durability**: `User is allergic to peanuts` is a
durable fact worth scarce core space; `Running: kubectl get pods` is
short-lived shell noise. Eviction is **not deletion** — evicted facts stay
searchable in archival. Below, four durable facts survive a flood of 16
short-lived shell/log lines, and an evicted short-lived fact is still findable.

In [ ]:
hm = load_skill_lib("hierarchical-memory")

mem = hm.HierarchicalMemory(core_limit=8)

# Day 1: durable operational facts
durables = [
    f"Production AWS account is {FICTIONAL_ACCOUNT}",
    "Primary region is us-east-1",
    "On-call rotation: Sarah primary, Marcus secondary",
    "Checkout API calls payments over synchronous HTTP",
]
for d in durables:
    mem.add_fact(d, hm.DURABILITY_DURABLE)

# Days 2-5: a flood of short-lived shell output and log tails
short_lived = [
    "Running: aws ec2 describe-instances", "Tail: 07:59 timeout",
    "Running: kubectl get pods", "Tail: 08:00 504 gateway",
    "Running: gh pr view 1234", "Tail: 08:01 503",
    "Running: terraform plan", "Tail: 08:02 retry-storm",
    "Running: aws logs filter-log-events", "Tail: 08:03 circuit-breaker open",
    "Running: psql -c count", "Tail: 08:04 db-connection-pool full",
    "Running: redis-cli ping", "Tail: 08:05 latency p99 8200ms",
    "Running: aws sts get-caller-identity", "Tail: 08:06 oncall-page-fired",
]
for s in short_lived:
    mem.add_fact(s, hm.DURABILITY_SHORT_LIVED)

diag = mem.diagnostics()
print(f"core size = {len(mem.core)} (limit 8), archival size = {len(mem.archival)}")

# Durable facts survive in core despite the flood
acct = mem.query("account")
print(f"\nquery('account') -> core hits: {[h['content'] for h in acct['core']]}")

# An evicted short-lived fact is still findable in archival (eviction != deletion)
cb = mem.query("circuit-breaker")
print(f"query('circuit-breaker') -> archival hits: "
      f"{[h['content'] for h in cb['archival']]}")

# --- Verification gate -----------------------------------------------------
survivors = [d for d in durables if d in mem.core]
assert len(survivors) >= 3, f"durable facts should resist eviction, only {len(survivors)} survived"
assert acct["core"], "the durable account fact should be a core hit"
assert cb["archival"], "an evicted short-lived fact must remain searchable in archival"
assert len(mem.core) <= 8, "core_limit must be honored"
print(f"\n[PASS] {len(survivors)}/4 durable facts stayed hot; evicted noise stayed searchable.")

## 4. Hindsight epistemic classifier — separate what the agent *observed* from what it *believes*

When a user asks *"how do you know that?"*, an agent that cannot distinguish
**evidence** from **inference** has no answer. The *Hindsight* paper (cited in
Chapter 4) organises memory into four networks:

- **World** — objective external facts (*the API returned 503s*).
- **Experience** — the agent's own first-person actions (*I called CloudWatch*).
- **Opinion** — subjective, hedged, confidence-scored beliefs (*I believe the
  root cause is v3.5.0*), carrying an `inferred_from` provenance chain.
- **Observation** — synthesized summaries (*the incident timeline is…*).

This separation is what lets the agent trace an *opinion* back to the
*world* and *experience* facts that produced it. Below, we classify the
post-mortem facts and then `justify` the root-cause opinion.

In [ ]:
he = load_skill_lib("hindsight-epistemic-classifier")

postmortem = [
    # World — objective
    {"text": "The checkout API returned 503 errors between 08:00 and 08:30 UTC.", "metadata": None},
    {"text": f"The production AWS account is {FICTIONAL_ACCOUNT}.",
     "metadata": {"external_ref": "iam-policy.json"}},
    # Experience — first-person agent actions (need a timestamp)
    {"text": "I called CloudWatch GetLogEvents at 08:15.",
     "metadata": {"action_type": "cloudwatch_call", "source": "agent",
                  "timestamp": datetime.now(timezone.utc).isoformat()}},
    {"text": "I retrieved the deployment log for v3.5.0.",
     "metadata": {"source": "agent", "timestamp": datetime.now(timezone.utc).isoformat()}},
    # Opinion — hedged belief with provenance
    {"text": "I believe the root cause is the v3.5.0 deployment.",
     "metadata": {"confidence": 0.7, "inferred_from": [
         "The checkout API returned 503 errors between 08:00 and 08:30 UTC.",
         "I retrieved the deployment log for v3.5.0."]}},
    # Observation — synthesized timeline from multiple sources
    {"text": "In summary, the timeline is: deploy 22:30, latency rising 07:45, 503s at 08:00, rollback 08:32.",
     "metadata": {"derived_from": ["deploy-log", "cloudwatch", "rollback-log", "incident-channel"]}},
]
facts = he.classify_batch(postmortem)
print("Epistemic classification:")
for f in facts:
    extra = f" conf={f.confidence}" if f.confidence < 1.0 else ""
    print(f"   [{f.network:11s}]{extra}  {f.text}")

audit = he.network_audit(facts)
print(f"\nnetwork_audit warnings: {audit['warnings'] or 'none'}")

# 'How do you know the root cause is v3.5.0?' -> trace the provenance
opinion_text = "I believe the root cause is the v3.5.0 deployment."
just = he.justify(facts, opinion_text)
print(f"\njustify('{opinion_text}'):")
print(json.dumps(just, indent=2))

# --- Verification gate -----------------------------------------------------
networks = {f.network for f in facts}
assert networks == {he.NETWORK_WORLD, he.NETWORK_EXPERIENCE, he.NETWORK_OPINION, he.NETWORK_OBSERVATION}, \
    f"all four epistemic networks should be represented, got {networks}"
assert just.get("provenance") and len(just["provenance"]) >= 2, \
    "the root-cause opinion must trace to >= 2 evidence facts"
print("\n[PASS] the agent can separate evidence from belief and justify its root-cause opinion.")

## 5. Letta failure modes — diagnose the architecture we just built

The Letta Leaderboard names **eight** failure modes that tend to co-occur:
unnecessary searches, broken hierarchies, in-conversation misses, degradation
at scale, silent overwrites, isolated silos, lost temporal coherence, and
collapse at thousands of facts. Chapter 4's whole design is the antidote.

This skill takes a `MemorySnapshot` (structural signals about a memory system)
and reports which failure modes are *present*, *warning*, *ok*, or *unknown*.
We run it via **`cli.py` as a subprocess** — the harness-portability path — on
two snapshots: the clean architecture we composed in cells 1-4, and a broken
anti-pattern for contrast.

In [ ]:
# Snapshot of the memory architecture we composed (bi-temporal on, tiered,
# extract_fn present, hybrid retrieval, connected graph) — should be clean.
clean_snapshot = {
    "core_size": len(mem.core), "core_limit": 8,
    "core_durable_count": sum(1 for f in mem.core.values() if f.durability == hm.DURABILITY_DURABLE),
    "core_short_lived_count": sum(1 for f in mem.core.values() if f.durability == hm.DURABILITY_SHORT_LIVED),
    "recall_size": 42, "archival_size": len(mem.archival), "archival_durable_count": 0,
    "node_count": len(g.nodes), "edge_count": len(g.edges),
    "disconnected_components": 2, "avg_degree": 3.4,
    "uses_bi_temporal_edges": True, "facts_without_created_at": 0,
    "facts_without_invalidation_reason_when_invalidated": 0,
    "retrieval_method": "hybrid", "retrieval_complexity_class": "O(log n)",
    "has_query_log": True, "extract_fn_present": True, "size_stress_test_passed": True,
}
# A broken architecture: overwrites in place, linear scan, disconnected silos.
broken_snapshot = {
    "core_size": 10, "core_limit": 10, "core_durable_count": 2, "core_short_lived_count": 8,
    "recall_size": 0, "archival_size": 200, "archival_durable_count": 15,
    "node_count": 50, "edge_count": 10, "disconnected_components": 30, "avg_degree": 0.4,
    "uses_bi_temporal_edges": False, "facts_without_created_at": 100,
    "facts_without_invalidation_reason_when_invalidated": 20,
    "retrieval_method": "linear-scan", "retrieval_complexity_class": "O(n)",
    "has_query_log": False, "extract_fn_present": False, "size_stress_test_passed": False,
}

tmpdir = Path(tempfile.mkdtemp(prefix="ch4_letta_"))
clean_path = tmpdir / "clean.json"
broken_path = tmpdir / "broken.json"
clean_path.write_text(json.dumps(clean_snapshot))
broken_path.write_text(json.dumps(broken_snapshot))

def diagnose_via_cli(path):
    proc = run_cli("letta-failure-modes", "diagnose", "--snapshot", str(path), "--format", "json")
    assert proc.returncode == 0, proc.stderr
    return json.loads(proc.stdout)

clean_report = diagnose_via_cli(clean_path)
broken_report = diagnose_via_cli(broken_path)
clean_present = [m for m in clean_report["modes"] if m["status"] == "present"]
broken_present = [m for m in broken_report["modes"] if m["status"] == "present"]
print(f"Clean architecture:  {len(clean_present)} failure modes present / {len(clean_report['modes'])}")
print(f"Broken architecture: {len(broken_present)} failure modes present / {len(broken_report['modes'])}")
print("\nFailure modes triggered by the broken architecture:")
for m in broken_present:
    print(f"   [sev {m['severity']}] {m['name']}: {m['fix']}")

# --- Verification gate -----------------------------------------------------
assert len(clean_report["modes"]) == 8, "there are exactly eight Letta failure modes"
assert len(clean_present) == 0, f"the composed architecture should trigger 0 failure modes, got {clean_present}"
assert len(broken_present) >= 6, "the broken anti-pattern should trigger most failure modes"
silent = next(m for m in broken_report["modes"] if m["name"] == "silent-overwrite")
assert silent["status"] == "present" and silent["severity"] == 3, "silent overwrite is a critical failure"
print("\n[PASS] the composed memory architecture is clean; the anti-pattern lights up.")

## 6. Memory consolidation — turn three noisy incidents into one durable **Pattern**

Raw episodes are too noisy to reason over directly. Consolidation (Example
4-5) is the agent's *sleep phase*: cluster related experiences, summarize each
cluster, and store the result as durable knowledge with a **provenance chain**
back to its sources — so *"how do you know this pattern?"* stays answerable.
The Letta/UC-Berkeley sleep-time-compute result the chapter cites: doing this
work during idle periods (not on the response path) is what makes it cheap.

The checkout API has had the *same* 503 / connection-pool-timeout shape after
three separate deploys. Consolidation collapses those three incidents into one
`Pattern`, keyed to all three source episodes, and pre-computes the at-risk
inference for the next deploy.

In [ ]:
mc = load_skill_lib("memory-consolidation")

records = [
    {"id": "ep1", "episode_type": "Incident",
     "content": "checkout-api 503 errors after deploy v3.5.0 raised connection pool timeout"},
    {"id": "ep2", "episode_type": "Incident",
     "content": "checkout-api 503 spike following deploy v3.6.1 connection pool exhausted timeout"},
    {"id": "ep3", "episode_type": "Incident",
     "content": "checkout-api 503 errors traced to deploy v3.7.0 connection pool timeout under load"},
    {"id": "ep4", "episode_type": "Deployment",
     "content": "routine deploy of inventory-service v2.1.0 no errors observed"},
    {"id": "ep5", "episode_type": "Conversation",
     "content": "user asked about billing dashboard color scheme preferences"},
]
episodes = mc.episodes_from_records(records)

clusters = mc.cluster_by_topic(episodes)
print("Topic clusters:", [[e.id for e in c] for c in clusters])

facts = mc.consolidate(episodes)  # min_cluster_size defaults to 3
print(f"\nConsolidated {len(facts)} durable fact(s):")
for fct in facts:
    print(f"   [{fct.knowledge_type}] {fct.summary}")
    print(f"      confirmations={fct.confirmations}  derived_from={fct.derived_from}")

# 'How do you know this pattern?' -> trace provenance to source episodes
print("\nProvenance of the consolidated pattern:")
for ep in mc.provenance_of(facts[0], episodes):
    print(f"   - {ep.id} [{ep.episode_type}]: {ep.content}")

# Sleep-time pre-computed inference (runs during idle periods, not on the response path)
inferences = mc.precompute_inferences(
    facts,
    inference_fn=lambda f: ["Risk: the next checkout-api deploy may exhaust the connection pool — pre-stage a runbook."],
)
print("\nSleep-time pre-computed inference:")
print(json.dumps(inferences, indent=2))

# --- Verification gate -----------------------------------------------------
assert len(facts) == 1, f"the three checkout incidents should consolidate to one pattern, got {len(facts)}"
assert facts[0].confirmations == 3, "the pattern should record three confirmations"
assert sorted(facts[0].derived_from) == ["ep1", "ep2", "ep3"], "provenance must list all three incident episodes"
assert "ep4" not in facts[0].derived_from and "ep5" not in facts[0].derived_from, \
    "unrelated episodes must not pollute the pattern"
print("\n[PASS] three noisy incidents consolidated into one provenance-backed Pattern.")

## 7. RRF hybrid retrieval — merge four channels into one ranked context

No single retrieval strategy catches everything. Chapter 4 (following Latimer
et al., 2025) runs four channels in parallel — **semantic**, **keyword**,
**graph traversal**, **temporal** — and merges them with **Reciprocal Rank
Fusion (RRF)**. RRF is rank-based (no score calibration across systems),
robust to missing items (absent = contributes nothing), and naturally floats
items that rank high in *multiple* channels. A cross-encoder rerank refines
the top candidates, then a token-budget filter fits the result into the LLM's
context window.

For our incident search, `INC-1042` (the checkout/v3.5.0 root-cause incident)
appears in all four channels, so RRF should surface it first.

In [ ]:
rr = load_skill_lib("rrf-hybrid-retrieval")

channels = {
    "semantic": ["INC-1043", "INC-1042", "INC-2024-11-23", "INC-1044"],
    "keyword":  ["INC-1042", "INC-1043", "INC-1044", "INC-1045"],
    "graph":    ["INC-1042", "deploy-v3.5.0", "service-payments-outage", "INC-1043"],
    "temporal": ["INC-1045", "deploy-v3.5.0", "INC-1044", "INC-1042"],
}
metadata = {
    "INC-1042": "Checkout API 503 errors caused by v3.5.0 deploy connection-pool timeouts on payments",
    "INC-1043": "Inventory service degradation traced to v2.1.0 deploy",
    "INC-1044": "Payments region-specific failover test",
    "INC-1045": "Fraud-detection service introduced as checkout dependency",
    "INC-2024-11-23": "Historical checkout latency incident from last year",
    "deploy-v3.5.0": "Deployment v3.5.0 of checkout-api at 22:30 UTC",
    "service-payments-outage": "Payments dependency outage entry",
}

fused = rr.rrf_fuse(channels)
print("RRF fusion (top 5, with channel coverage):")
for doc_id, score in fused[:5]:
    coverage = sum(1 for c in channels.values() if doc_id in c)
    print(f"   {score:.5f}  {doc_id:24s} (in {coverage}/4 channels)")

reranked = rr.cross_encoder_rerank(fused[:6], query="checkout latency from a recent deploy", metadata=metadata)
print("\nCross-encoder rerank (top 3):")
for doc_id, fs, rs in reranked[:3]:
    print(f"   fusion={fs:.5f} rerank={rs:.3f}  {doc_id}")

final = rr.token_budget_filter(reranked[:5], get_tokens=lambda d: len(metadata.get(d, d)), budget=200)
print(f"\nToken-budget-filtered context ({len(final)} items under a 200-token budget):")
for item in final:
    print(f"   {item['doc_id']}  ({item['token_count']} tokens)")

# --- Verification gate -----------------------------------------------------
assert fused[0][0] == "INC-1042", "the incident in all four channels should fuse to the top"
assert sum(item["token_count"] for item in final) <= 200, "the token budget must be respected"
top_reranked = {r[0] for r in reranked[:3]}
assert "INC-1042" in top_reranked, "the root-cause incident should survive reranking into the top-3"
print("\n[PASS] four retrieval channels fused, reranked, and budget-fitted — root-cause incident on top.")

## 8. Verdict — the temporal root-cause query the memory can now answer

Chapter 3 gave the agent a *snapshot*. After threading one incident dataset
through all seven memory primitives, the agent can answer the question the
snapshot never could:

> *"At 2026-03-15T08:00Z the checkout API went down. What was running, what
> changed just before, is this a known pattern, and how do you know?"*

The final cell composes the seven primitives into that one answer:

1. **bi-temporal** `as_of(outage)` reconstructs the exact config at outage time.
2. **graphiti** supplies the entity graph the incident touched.
3. **hierarchical** kept the durable operational facts hot.
4. **hindsight** separates the *observed* evidence from the *believed* cause.
5. **letta-failure-modes** confirmed the architecture is clean.
6. **consolidation** recognises the recurring **Pattern**.
7. **rrf** ranked the root-cause incident to the top of retrieval.

In [ ]:
print("=" * 72)
print(f"INCIDENT ROOT-CAUSE ANSWER  —  outage {OUTAGE_TS.isoformat()}  —  account {FICTIONAL_ACCOUNT}")
print("=" * 72)

# 1. What was running at outage time? (bi-temporal as_of)
inst = bt.as_of("service-checkout-api", "instance_type", OUTAGE_TS, edges).value
dep = bt.as_of("service-checkout-api", "deployment_version", OUTAGE_TS, edges).value
print(f"\n1. State AT outage time (bi-temporal reconstruction):")
print(f"     instance_type = {inst}   deployment_version = {dep}")

# 2. What changed in the 24h before? (bi-temporal change window)
changes = [e for e in bt.history("service-checkout-api", edges)
           if OUTAGE_TS - timedelta(hours=24) <= e.valid_from < OUTAGE_TS]
print(f"\n2. Change in the 24h before the outage:")
for e in changes:
    print(f"     {e.relationship} -> {e.value}  at {e.valid_from.isoformat()}")

# 3. Is this a known pattern? (consolidation)
pattern = facts[0]
print(f"\n3. Known pattern (consolidated from {pattern.confirmations} prior incidents):")
print(f"     {pattern.summary}")

# 4. How do you know the cause? (hindsight provenance)
print(f"\n4. Evidence for the root-cause belief (hindsight provenance):")
for src in just["provenance"]:
    print(f"     - {src}")

# 5. Top retrieved incident (rrf) + architecture health (letta)
print(f"\n5. Top retrieved incident: {fused[0][0]}  |  memory-architecture failure modes: {len(clean_present)}/8")

# Compose the one-sentence conclusion and verify it is internally consistent
conclusion = (
    f"The {inst} checkout-api was stable, but deployment {dep} shipped in the 24h "
    f"before the 08:00 outage; this matches a known connection-pool-timeout pattern "
    f"seen in {pattern.confirmations} prior deploys."
)
print("\n" + "-" * 72)
print("CONCLUSION:", conclusion)
print("-" * 72)

# --- Final verification: the composed answer is coherent across all 7 skills
assert dep == "v3.5.0" and any(e.value == "v3.5.0" for e in changes), "the deploy is both current-at-outage and the recent change"
assert pattern.confirmations == 3, "the pattern generalises three incidents"
assert len(just["provenance"]) >= 2, "the root-cause belief is evidence-backed"
assert fused[0][0] == "INC-1042", "retrieval surfaces the root-cause incident"
assert len(clean_present) == 0, "the memory architecture is clean"
print("\nALL GATES PASSED — the DevOps agent's memory answers the temporal root-cause query end-to-end.")